# Notebook 03: Approval Funnel
**Objective:** Analyse approval cycle times, identify bottlenecks by segment and approver, and understand what drives deal rejection.

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
deals_df = pd.read_csv('../data/raw/deals.csv')
approvals_df = pd.read_csv('../data/raw/approvals.csv')
outcomes_df = pd.read_csv('../data/raw/outcomes.csv')

deals_df['created_date'] = pd.to_datetime(deals_df['created_date'])
approvals_df['submitted_date'] = pd.to_datetime(approvals_df['submitted_date'])
approvals_df['decision_date'] = pd.to_datetime(approvals_df['decision_date'])
outcomes_df['close_date'] = pd.to_datetime(outcomes_df['close_date'])

con = duckdb.connect()
con.register('deals', deals_df)
con.register('approvals', approvals_df)
con.register('outcomes', outcomes_df)

print(f"deals:     {len(deals_df):,} rows")
print(f"approvals: {len(approvals_df):,} rows")
print(f"outcomes:  {len(outcomes_df):,} rows")

## Approval Funnel Overview

In [ ]:
funnel = con.execute("""
    SELECT
        COUNT(DISTINCT d.deal_id)                               AS total_deals,
        COUNT(DISTINCT a.deal_id)                               AS deals_requiring_approval,
        SUM(CASE WHEN a.status = 'Approved'  THEN 1 ELSE 0 END) AS approved,
        SUM(CASE WHEN a.status = 'Rejected'  THEN 1 ELSE 0 END) AS rejected,
        SUM(CASE WHEN a.status = 'Pending'   THEN 1 ELSE 0 END) AS pending,
        ROUND(
            COUNT(DISTINCT a.deal_id) * 100.0 / COUNT(DISTINCT d.deal_id), 1
        ) AS pct_requiring_approval
    FROM deals d
    LEFT JOIN approvals a ON d.deal_id = a.deal_id
""").df()

row = funnel.iloc[0]
print(f"Total deals:                {row['total_deals']:>8,.0f}")
print(f"Requiring approval:         {row['deals_requiring_approval']:>8,.0f}  ({row['pct_requiring_approval']:.1f}%)")
print(f"  Approved:                 {row['approved']:>8,.0f}")
print(f"  Rejected:                 {row['rejected']:>8,.0f}")
print(f"  Pending:                  {row['pending']:>8,.0f}")

## Approval Cycle Time by Segment

In [ ]:
cycle_time = con.execute("""
    SELECT
        d.segment,
        COUNT(*) AS total_approvals,
        ROUND(AVG(a.cycle_days), 1) AS avg_days,
        ROUND(MIN(a.cycle_days), 1) AS min_days,
        ROUND(MAX(a.cycle_days), 1) AS max_days,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY a.cycle_days) AS median_days,
        PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY a.cycle_days) AS p90_days
    FROM approvals a
    JOIN deals d ON a.deal_id = d.deal_id
    WHERE a.cycle_days IS NOT NULL
    GROUP BY d.segment
    ORDER BY avg_days DESC
""").df()

print(
    f"{'Segment':<15} {'Count':>8} {'Avg':>8} {'Median':>8} "
    f"{'P90':>8} {'Min':>8} {'Max':>8}"
)
print("-" * 65)
for _, row in cycle_time.iterrows():
    print(
        f"{row['segment']:<15} {row['total_approvals']:>8,.0f} "
        f"{row['avg_days']:>7.1f}d {row['median_days']:>7.1f}d "
        f"{row['p90_days']:>7.1f}d {row['min_days']:>7.1f}d "
        f"{row['max_days']:>7.1f}d"
    )

## Cycle Time Distribution by Segment

In [ ]:
segments = ['SMB', 'Mid-Market', 'Enterprise']
colors = ['#636EFA', '#EF553B', '#00CC96']

approved_df = approvals_df.merge(
    deals_df[['deal_id', 'segment']], on='deal_id'
).dropna(subset=['cycle_days'])

fig = go.Figure()

for seg, color in zip(segments, colors):
    d = approved_df[approved_df['segment'] == seg]['cycle_days']
    fig.add_trace(go.Box(
        y=d,
        name=seg,
        marker_color=color,
        boxmean=True
    ))

fig.update_layout(
    title='Approval Cycle Time by Segment (days)',
    yaxis_title='Cycle Days',
    yaxis=dict(range=[0, 25]),
    height=500,
    font=dict(size=13, color='#212121'),
    template='plotly_white'
)

fig.show()
fig.write_image('../outputs/05_cycle_time_by_segment.png')

## Approval Cycle Time vs Win Rate

In [ ]:
cycle_outcome = con.execute("""
    SELECT
        a.cycle_days,
        o.win_flag,
        d.deal_type
    FROM approvals a
    JOIN deals d ON a.deal_id = d.deal_id
    JOIN outcomes o ON a.deal_id = o.deal_id
    WHERE a.cycle_days IS NOT NULL
""").df()

bins = [0, 2, 5, 8, 20]
labels = ['0-2 days', '3-5 days', '6-8 days', '8+ days']

cycle_outcome['cycle_band'] = pd.cut(
    cycle_outcome['cycle_days'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

for deal_type in ['New Business', 'Expansion', 'Renewal']:
    d = cycle_outcome[cycle_outcome['deal_type'] == deal_type]
    band_wr = (
        d.groupby('cycle_band', observed=True)
        .agg(
            deals=('win_flag', 'count'),
            win_rate=('win_flag', 'mean')
        )
        .reset_index()
    )

    print(f"\n--- {deal_type} ---")
    print(f"{'Cycle Band':<12} {'Deals':>8} {'Win Rate':>10}")
    print("-" * 32)
    for _, row in band_wr.iterrows():
        print(
            f"{str(row['cycle_band']):<12} {row['deals']:>8,} "
            f"{row['win_rate']:>9.1%}"
        )

## Cycle Time vs Win Rate Chart

In [ ]:
nb_cycle = cycle_outcome[cycle_outcome['deal_type'] == 'New Business']

band_wr_nb = (
    nb_cycle.groupby('cycle_band', observed=True)
    .agg(
        deals=('win_flag', 'count'),
        win_rate=('win_flag', 'mean')
    )
    .reset_index()
)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=band_wr_nb['cycle_band'].astype(str),
    y=band_wr_nb['win_rate'],
    text=[f"{v:.1%}" for v in band_wr_nb['win_rate']],
    textposition='outside',
    marker_color='#636EFA'
))

fig.update_layout(
    title='Win Rate by Approval Cycle Time — New Business Only',
    xaxis_title='Approval Cycle Band',
    yaxis_title='Win Rate',
    yaxis=dict(tickformat='.0%', range=[0, 0.80]),
    height=500,
    font=dict(size=13, color='#212121'),
    template='plotly_white'
)

fig.show()
fig.write_image('../outputs/06_winrate_by_cycle_time.png')

## Rejection Analysis

In [ ]:
rejections = con.execute("""
    SELECT
        a.rejection_reason,
        COUNT(*) AS total,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM approvals a
    WHERE a.status = 'Rejected'
    GROUP BY a.rejection_reason
    ORDER BY total DESC
""").df()

print(f"{'Rejection Reason':<35} {'Count':>8} {'Pct':>8}")
print("-" * 53)
for _, row in rejections.iterrows():
    print(
        f"{row['rejection_reason']:<35} {row['total']:>8,} "
        f"{row['pct']:>7.1f}%"
    )

## Rejection Reasons Chart

In [ ]:
rejections_sorted = rejections.sort_values('total', ascending=True)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=rejections_sorted['total'],
    y=rejections_sorted['rejection_reason'],
    orientation='h',
    text=[f"{v:.1f}%" for v in rejections_sorted['pct']],
    textposition='outside',
    marker_color='#EF553B'
))

fig.update_layout(
    title='Deal Rejection Reasons',
    xaxis_title='Number of Rejections',
    xaxis=dict(range=[0, 35]),
    height=450,
    font=dict(size=13, color='#212121'),
    template='plotly_white'
)

fig.show()
fig.write_image('../outputs/07_rejection_reasons.png')

## Rejection Rate by Segment and Approver

In [ ]:
rejection_breakdown = con.execute("""
    SELECT
        d.segment,
        a.approver,
        COUNT(*) AS total_approvals,
        SUM(CASE WHEN a.status = 'Rejected' THEN 1 ELSE 0 END) AS rejections,
        ROUND(
            SUM(CASE WHEN a.status = 'Rejected' THEN 1 ELSE 0 END) * 100.0
            / COUNT(*), 1
        ) AS rejection_rate
    FROM approvals a
    JOIN deals d ON a.deal_id = d.deal_id
    GROUP BY d.segment, a.approver
    ORDER BY d.segment, rejection_rate DESC
""").df()

for seg in ['SMB', 'Mid-Market', 'Enterprise']:
    d = rejection_breakdown[rejection_breakdown['segment'] == seg]
    print(f"\n--- {seg} ---")
    print(f"{'Approver':<12} {'Approvals':>12} {'Rejections':>12} {'Rej Rate':>10}")
    print("-" * 48)
    for _, row in d.iterrows():
        print(
            f"{row['approver']:<12} {row['total_approvals']:>12,} "
            f"{row['rejections']:>12,} {row['rejection_rate']:>9.1f}%"
        )

## Key Findings

- 33% of deals (660) required approval; 82.0% approved, 12.7% rejected, 5.3% pending
- Enterprise approvals average 8 days; SMB averages 1.6 days — 5x slower
- New Business deals approved within 2 days win 54.8% vs 40.8% for 3-5 days — a 14 point gap
- Cycle time impact is not significant for Expansions and Renewals — pre-approval policies warranted
- Discount-related reasons (exceeds threshold + margin too low) account for 48.8% of all rejections
- VP Sales is the strictest approver — highest rejection rate across SMB (16.9%) and Mid-Market (13.3%)
- CFO is most lenient on Enterprise deals at 3.7% rejection rate — likely due to pre-vetting
- SLA recommendation: SMB 2 days, Mid-Market 5 days, Enterprise 10 days based on P90 cycle times